In [23]:
import os

print("=== /kaggle/input ===")
for d in os.listdir("/kaggle/input"):
    print(" •", d)

=== /kaggle/input ===
 • datasets


In [24]:
import json, os

# Recherche automatique des fichiers sur Kaggle
dossier_splits = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.startswith("kb_manifest") and f.endswith(".json"):
            dossier_splits = root
            nom_kb = f
            break
    if dossier_splits:
        break

if not dossier_splits:
    print("❌ Impossible de trouver les fichiers sur Kaggle.")
else:
    print(f"✅ Fichiers trouvés dans : {dossier_splits}")
    fichiers = os.listdir(dossier_splits)
    nom_kb = [f for f in fichiers if f.startswith("kb_manifest")][0]
    nom_val = [f for f in fichiers if f.startswith("val_manifest")][0]
    nom_test = [f for f in fichiers if f.startswith("test_manifest")][0]
    
    kb   = json.load(open(f"{dossier_splits}/{nom_kb}"))
    val  = json.load(open(f"{dossier_splits}/{nom_val}"))
    test = json.load(open(f"{dossier_splits}/{nom_test}"))

    print(f"\nKB={len(kb)} | Val={len(val)} | Test={len(test)} | TOTAL={len(kb)+len(val)+len(test)}")

✅ Fichiers trouvés dans : /kaggle/input/datasets/malakbenalikhodja/splits-member1

KB=10000 | Val=100 | Test=1000 | TOTAL=11100


In [25]:
import re

def clean(c):
    c = re.sub(r"[^a-z0-9\s']", " ", c.lower())
    return re.sub(r"\s+", " ", c).strip()

def build(items, split):
    return [{
        "image_id": x["image_id"],
        "split": split,
        "captions_raw":   x["captions"],
        "captions_clean": [clean(c) for c in x["captions"]],
        "tokens":         [clean(c).split() for c in x["captions"]],
    } for x in items]

caps = build(kb,"kb") + build(val,"val") + build(test,"test")
json.dump(caps, open("captions_processed.json","w"), indent=2)
print(f"✅ {len(caps)} entrées → captions_processed.json")

✅ 11100 entrées → captions_processed.json


In [26]:
!pip -q install ultralytics
import ultralytics
from ultralytics import YOLO
from PIL import Image, ImageOps
from tqdm.notebook import tqdm
import json
import os

# 1. Chargement du modèle
print("⏳ Chargement du modèle YOLOv8m...")
model = YOLO("yolov8m.pt")
CONF = 0.25
root = "/kaggle/input/datasets/sabahesaraki/2017-2017"

def get_path(x, split):
    folder = "train2017" if split in ("kb","val") else "val2017"
    p1 = f"{root}/{folder}/{folder}/{x['file_name']}"
    return p1 if os.path.exists(p1) else f"{root}/{folder}/{x['file_name']}"

# 2. Préparation de la liste des images
items = [(x,"kb") for x in kb] + [(x,"val") for x in val] + [(x,"test") for x in test]
paths = [get_path(x,s) for x,s in items]

meta = []
print("⏳ Lancement de YOLO. Traitement image par image (environ 10-15 minutes)...")

# 3. Traitement image par image (beaucoup plus stable pour Kaggle)
for (x, s), p in tqdm(zip(items, paths), total=len(items), desc="Traitement"):
    # On donne UNE SEULE image au modèle à la fois
    r = model.predict(p, conf=CONF, verbose=False, device=0)[0]
    
    objs = [{
        "label": r.names[int(c)],
        "conf":  round(float(p), 3),
        "bbox":  [round(float(v), 1) for v in b],
    } for c, p, b in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxy)]
    
    meta.append({"image_id": x["image_id"], "split": s, "objects": objs})

# 4. Sauvegarde du fichier final
json.dump({
    "detector": "YOLOv8m",
    "ultralytics_version": ultralytics.__version__,
    "conf_threshold": CONF,
    "data": meta,
}, open("metadata.json","w"), indent=2)

print(f"\n✅ TERMINÉ ! {len(meta)} images traitées et sauvegardées dans metadata.json")

⏳ Chargement du modèle YOLOv8m...
⏳ Lancement de YOLO. Traitement image par image (environ 10-15 minutes)...


Traitement:   0%|          | 0/11100 [00:00<?, ?it/s]


✅ TERMINÉ ! 11100 images traitées et sauvegardées dans metadata.json


In [27]:
assert len(meta) == 11100, f"Erreur : {len(meta)} images au lieu de 11100"
no_obj = sum(1 for m in meta if not m["objects"])
tot    = sum(len(m["objects"]) for m in meta)
print(f"Images sans objet : {no_obj}")
print(f"Moyenne d'objets/image : {tot/len(meta):.2f}")
print("✅ Vérification réussie")

Images sans objet : 88
Moyenne d'objets/image : 7.06
✅ Vérification réussie


In [28]:
import os
# On teste la toute première image de ta liste
test_path = paths[0] 
print("Chemin testé :", test_path)
print("Est-ce que l'image existe ? :", os.path.exists(test_path))

Chemin testé : /kaggle/input/datasets/sabahesaraki/2017-2017/train2017/train2017/000000000009.jpg
Est-ce que l'image existe ? : True


In [29]:
import torch
print("GPU disponible :", torch.cuda.is_available())
print("Nom du GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")

GPU disponible : True
Nom du GPU : Tesla T4
